# 03 · Secondary task: guest location rating

`review_scores_location` is set by guests and depends almost only on where the listing is. It is the clearest test of whether the model breaks down without external data. Sample: listings with at least 10 reviews, September 2026 snapshot.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
from config import CLASSMATE_REPO, SLIM, EXT, OUT
from common import *
from external import build
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
st = pd.read_csv(CLASSMATE_REPO / "data" / "stations_used.csv")   # official IDFM stations (classmate's repo)
print("classmate repo:", CLASSMATE_REPO.exists(), "| slim:", SLIM.exists(), "| external:", EXT.exists())

classmate repo: True | slim: True | external: True


In [2]:
L = pd.read_csv(SLIM / "listings_slim.csv.gz", low_memory=False)
d = L[(L.number_of_reviews >= 10) & L.review_scores_location.notna()].reset_index(drop=True)
d["bathrooms_n"] = pd.to_numeric(d.bathrooms_text.astype(str).str.extract(r"([\d.]+)")[0], errors="coerce")
d["amenity_count"] = d.amenities.astype(str).str.count('",') + 1
X, G, info, _ = build(d.latitude.values, d.longitude.values, EXT, st); d = pd.concat([d, X], axis=1)
d["cell"] = cell_1km(d.latitude, d.longitude); d["block"] = block_3km(d.latitude, d.longitude)
cat(d, ["room_type", "property_type", "neighbourhood_cleansed"])
print(len(d), "listings | rating mean", d.review_scores_location.mean().round(3), "sd", d.review_scores_location.std().round(3))

38594 listings | rating mean 4.847 sd 0.17


In [3]:
A = ["accommodates", "bedrooms", "beds", "bathrooms_n", "minimum_nights", "amenity_count", "room_type", "property_type", "calculated_host_listings_count"]
EXT_ALL = sum(G.values(), []); GEO = ["latitude", "longitude", "neighbourhood_cleansed"]
S = {"A internal": A, **{f"A+{k}": A + v for k, v in G.items()}, "B A+all external": A + EXT_ALL,
     **{f"B-{k}": A + [c for c in EXT_ALL if c not in G[k]] for k in G}, "C A+coordinates": A + GEO, "D C+all external": A + GEO + EXT_ALL}
rows = []
run_task("T8 location rating", d, d.review_scores_location.values, "reg", S, rows, protocols=("host", "spatial", "block"),
         only={"block": ["A internal", "B A+all external", "C A+coordinates", "D C+all external"]})
t8 = pd.DataFrame(rows); t8.to_csv(OUT / "t8_rating.csv", index=False)
t8.pivot_table(index="features", columns="protocol", values="R2", sort=False)[["host", "spatial", "block"]].round(3)

T8 location rating           host     A internal                         R2=0.0234  MAE=0.1160


T8 location rating           host     A+stations                         R2=0.3358  MAE=0.0941


T8 location rating           host     A+osm                              R2=0.5080  MAE=0.0801


T8 location rating           host     A+dvf                              R2=0.4683  MAE=0.0830


T8 location rating           host     A+iris_income                      R2=0.4492  MAE=0.0853


T8 location rating           host     A+crime                            R2=0.2279  MAE=0.0997


... (23 outputs left out)

T8 location rating           block    B A+all external                   R2=0.4288  MAE=0.0848


T8 location rating           block    C A+coordinates                    R2=0.1586  MAE=0.1027


T8 location rating           block    D C+all external                   R2=0.3970  MAE=0.0858


protocol,host,spatial,block
features,,,
A internal,0.023,-0.007,-0.045
A+stations,0.336,0.081,NaN
A+osm,0.508,0.389,NaN
A+dvf,0.468,0.394,NaN
A+iris_income,0.449,0.317,NaN
A+crime,0.228,0.132,NaN
B A+all external,0.522,0.476,0.429
B-stations,0.520,0.471,NaN
B-osm,0.516,0.477,NaN


In [4]:
s = t8[t8.protocol == "block"].set_index("features").folds.str.split(";").apply(lambda x: np.array(x, float))
pd.DataFrame({"A→B": s["B A+all external"] - s["A internal"], "C→B": s["B A+all external"] - s["C A+coordinates"]}).round(3)

,A→B,C→B
0,0.345,0.321
1,1.015,0.205
2,0.364,0.098
3,0.217,-0.023
4,0.675,0.460
